# 3. Compute front velocities: left and right

Use this only after the density/profile validation looks reasonable.

The left front position usually decreases with time, so its raw fitted slope `dx/dt` is negative. For comparison, this notebook reports the **outward speed**:

- left outward speed = `- dx_left/dt`
- right outward speed = `+ dx_right/dt`

The FKPP Brownian estimate uses the dilute-edge growth rate `r = p0 - q0`: `v_FKPP = 2 sqrt(Dr r)`. For reference, the notebook also prints an active-effective estimate using `D_eff = Dr + v0^2/(2 Dtheta)`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from front_analysis import (
    get_run_files,
    discover_run_ids,
    read_trajectory,
    read_front_file,
    plot_front_timeseries_lr,
    plot_front_fit_side,
    velocity_summary_table,
    plot_velocity_comparison,
    fkpp_estimates,
)

param_base = "params_brownian_Dr_sweep_glued"
data_folder = Path("data") / param_base
run_id = 132

t_min = 0.0
t_max = None  # None will be replaced by the final saved front time.

methods = ["th_1", "th_2", "th_3"]
main_method = "th_2"

output_folder = Path("figures") / param_base / "velocities"
output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
threshold_methods = ["th_1", "th_2", "th_3"]
main_method_default = "th_2"

## 1. Load one run and inspect the left/right time series

In [ ]:
dat_file, front_file, rho_file = get_run_files(data_folder, param_base, run_id)
params, frames = read_trajectory(dat_file)
front_meta, front = read_front_file(front_file)

if t_max is None:
    t_max = float(front["time"][-1])

print("run_id:", run_id)
print("fit interval:", t_min, "to", t_max)
print("rho_sat_used:", front_meta.get("rho_sat_used"))

plot_front_timeseries_lr(
    front,
    methods=methods,
    t_min=None,
    t_max=None,
    save_path=output_folder / f"run_{run_id:03d}_front_timeseries_lr.png",
    show=True,
)

## 2. Fit the main threshold method on left and right

In [ ]:
fit_start_fraction = 0.70
fit_end_fraction = 0.95

t_min =fit_start_fraction*front["time"].max()
t_max = fit_end_fraction*front["time"].max()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)

plot_front_fit_side(front, "left",  main_method, t_min, t_max, ax=axes[0], show=False)
plot_front_fit_side(front, "right", main_method, t_min, t_max, ax=axes[1], show=False)

axes[0].set_title("left front")
axes[1].set_title("right front")

axes[1].set_ylabel("")

axes[0].text(0.03, 0.95, "(a)", transform=axes[0].transAxes, va="top")
axes[1].text(0.03, 0.95, "(b)", transform=axes[1].transAxes, va="top")

fig.tight_layout()
fig.savefig(output_folder / f"run_{run_id:03d}_{main_method}_fit_lr_1x2.png",
            dpi=160, bbox_inches="tight")
plt.show()

## 3. Compare velocities from all methods

In [ ]:
rows = velocity_summary_table(front, t_min=t_min, t_max=t_max, methods=methods)
velocity_df = pd.DataFrame(rows)
display(velocity_df)

plot_velocity_comparison(
    rows,
    save_path=output_folder / f"run_{run_id:03d}_velocity_comparison_lr.png",
    show=True,
)

## 4. FKPP reference estimate

This is not expected to be exact for the active interacting model. It is a reference scale.

In [ ]:
combined_metadata = {}
combined_metadata.update(params)
combined_metadata.update(front_meta)

fkpp = fkpp_estimates(combined_metadata)
for key, value in fkpp.items():
    print(f"{key:20s} = {value}")

main_row = velocity_df[velocity_df["method"] == main_method].iloc[0]
print()
print("Measured main-method speeds:")
print("left outward speed :", main_row["left_outward_speed"])
print("right outward speed:", main_row["right_outward_speed"])
print("mean outward speed :", main_row["mean_outward_speed"])
print("Brownian FKPP      :", fkpp["v_fkpp_brownian"])
print("Active-eff FKPP    :", fkpp["v_fkpp_active_eff"])

## 5. Batch velocity table across runs

In [ ]:
run_ids = discover_run_ids(data_folder, param_base, source="front")
all_rows = []

for rid in run_ids:
    dat_path, front_path, rho_path = get_run_files(data_folder, param_base, rid)
    if not Path(front_path).exists():
        continue
    fmeta, fdata = read_front_file(front_path)
    local_tmax = t_max
    if local_tmax is None:
        local_tmax = float(fdata["time"][-1])
    for row in velocity_summary_table(fdata, t_min=t_min, t_max=local_tmax, methods=methods):
        row["run_id"] = rid
        row["rho_sat_used"] = fmeta.get("rho_sat_used", np.nan)
        all_rows.append(row)

all_velocity_df = pd.DataFrame(all_rows)
display(all_velocity_df)

In [ ]:
if len(all_velocity_df) > 0:
    summary = (
        all_velocity_df
        .groupby("method")[["left_outward_speed", "right_outward_speed", "mean_outward_speed", "relative_asymmetry"]]
        .agg(["mean", "std", "count"])
    )
    display(summary)

In [ ]:
from front_analysis import get_front_column, side_sign

def fit_linearity_diagnostics(front_data, side="right", method="th_2",
                              t_min=None, t_max=None,
                              exclude_boundary_hit=True):
    """
    Check whether one front is well described by a straight line:
        x_front(t) = intercept + slope * t

    For the left front, outward speed = -slope.
    For the right front, outward speed = +slope.
    """
    col = get_front_column(side, method)

    t = np.asarray(front_data["time"], dtype=float)
    x = np.asarray(front_data[col], dtype=float)

    mask = np.isfinite(t) & np.isfinite(x)

    if exclude_boundary_hit and "hit_boundary" in front_data:
        mask &= np.asarray(front_data["hit_boundary"]) < 0.5

    if t_min is not None:
        mask &= t >= t_min
    if t_max is not None:
        mask &= t <= t_max

    t_fit = t[mask]
    x_fit_data = x[mask]

    if len(t_fit) < 5:
        raise ValueError("Not enough points in the selected fitting interval.")

    slope, intercept = np.polyfit(t_fit, x_fit_data, 1)
    x_fit = slope * t_fit + intercept
    residuals = x_fit_data - x_fit

    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((x_fit_data - np.mean(x_fit_data))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    rmse = np.sqrt(np.mean(residuals**2))

    outward_speed = side_sign(side) * slope

    return {
        "side": side,
        "method": method,
        "column": col,
        "t": t,
        "x": x,
        "mask": mask,
        "t_fit": t_fit,
        "x_fit_data": x_fit_data,
        "slope": slope,
        "intercept": intercept,
        "outward_speed": outward_speed,
        "residuals": residuals,
        "r2": r2,
        "rmse": rmse,
    }


def plot_linearity_check(front_data, method="th_2",
                         t_min=None, t_max=None,
                         exclude_boundary_hit=True):
    """
    Plot front position with linear fit and residuals for left and right fronts.
    """
    results = []

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex="col")

    for i, side in enumerate(["left", "right"]):
        res = fit_linearity_diagnostics(
            front_data,
            side=side,
            method=method,
            t_min=t_min,
            t_max=t_max,
            exclude_boundary_hit=exclude_boundary_hit,
        )
        results.append(res)

        t = res["t"]
        x = res["x"]
        mask = res["mask"]
        t_fit = res["t_fit"]
        x_line = res["slope"] * t_fit + res["intercept"]

        ax = axes[i, 0]
        ax.plot(t, x, ".", markersize=4, label="front data")
        ax.plot(t_fit, x_line, "-", linewidth=2,
                label=fr"fit, $v_{{out}}={res['outward_speed']:.5g}$")
        ax.axvline(t_fit[0], linestyle=":", linewidth=1)
        ax.axvline(t_fit[-1], linestyle=":", linewidth=1)
        ax.set_ylabel(fr"{side} front $x(t)$")
        ax.legend(fontsize=9)

        ax = axes[i, 1]
        ax.axhline(0.0, linestyle="--", linewidth=1)
        ax.plot(t_fit, res["residuals"], "o-", markersize=4)
        ax.set_ylabel("residual")
        ax.set_title(fr"{side}: $R^2={res['r2']:.6f}$, RMSE={res['rmse']:.4g}")

    axes[1, 0].set_xlabel("time")
    axes[1, 1].set_xlabel("time")

    fig.suptitle(fr"Linearity check for {method}")
    fig.tight_layout()
    plt.show()

    return pd.DataFrame([
        {
            "side": r["side"],
            "method": r["method"],
            "slope_dxdt": r["slope"],
            "outward_speed": r["outward_speed"],
            "R2": r["r2"],
            "RMSE": r["rmse"],
            "n_points": len(r["t_fit"]),
            "t_start": r["t_fit"][0],
            "t_end": r["t_fit"][-1],
        }
        for r in results
    ])


linearity_df = plot_linearity_check(
    front,
    method=main_method,
    t_min=t_min,
    t_max=t_max,
    exclude_boundary_hit=True,
)

display(linearity_df)

In [ ]:
def sliding_window_velocity(front_data, side="right", method="th_2",
                            window_fraction=0.15,
                            exclude_boundary_hit=True):
    """
    Compute local front velocity by fitting small moving windows.

    window_fraction = fraction of the clean total time used in each local fit.
    For example, 0.15 means each local fit uses 15% of the trajectory duration.
    """
    col = get_front_column(side, method)

    t = np.asarray(front_data["time"], dtype=float)
    x = np.asarray(front_data[col], dtype=float)

    mask = np.isfinite(t) & np.isfinite(x)

    if exclude_boundary_hit and "hit_boundary" in front_data:
        mask &= np.asarray(front_data["hit_boundary"]) < 0.5

    t = t[mask]
    x = x[mask]

    if len(t) < 8:
        return pd.DataFrame()

    total_time = t[-1] - t[0]
    window_width = window_fraction * total_time

    rows = []

    for center in t:
        local_mask = (t >= center - 0.5 * window_width) & (t <= center + 0.5 * window_width)

        if np.sum(local_mask) < 5:
            continue

        slope, intercept = np.polyfit(t[local_mask], x[local_mask], 1)
        outward_speed = side_sign(side) * slope

        rows.append({
            "side": side,
            "method": method,
            "t_center": center,
            "outward_speed": outward_speed,
            "slope_dxdt": slope,
            "n_points": int(np.sum(local_mask)),
        })

    return pd.DataFrame(rows)


left_local = sliding_window_velocity(front, "left", main_method, window_fraction=0.15)
right_local = sliding_window_velocity(front, "right", main_method, window_fraction=0.15)

local_df = pd.concat([left_local, right_local], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 4.8))

for side, group in local_df.groupby("side"):
    ax.plot(group["t_center"], group["outward_speed"], marker="o", linestyle="-", label=side)

ax.set_xlabel("time")
ax.set_ylabel("local outward speed")
ax.set_title(fr"Sliding-window velocity, method={main_method}")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

display(local_df.head())

In [ ]:
def fit_one_fraction_window(front_data, method="th_2",
                            start_fraction=0.70,
                            end_fraction=0.9,
                            exclude_boundary_hit=True):
    """
    Fit left and right fronts using a fractional time window.
    Fractions are computed after optionally removing the boundary-hit row.
    """
    t = np.asarray(front_data["time"], dtype=float)
    clean_mask = np.isfinite(t)

    if exclude_boundary_hit and "hit_boundary" in front_data:
        clean_mask &= np.asarray(front_data["hit_boundary"]) < 0.5

    t_clean = t[clean_mask]
    t0 = t_clean[0]
    t1 = t_clean[-1]

    local_t_min = t0 + start_fraction * (t1 - t0)
    local_t_max = t0 + end_fraction * (t1 - t0)

    rows = []

    for side in ["left", "right"]:
        res = fit_linearity_diagnostics(
            front_data,
            side=side,
            method=method,
            t_min=local_t_min,
            t_max=local_t_max,
            exclude_boundary_hit=exclude_boundary_hit,
        )

        rows.append({
            "window": f"{start_fraction:.2f}-{end_fraction:.2f}",
            "start_fraction": start_fraction,
            "end_fraction": end_fraction,
            "side": side,
            "outward_speed": res["outward_speed"],
            "R2": res["r2"],
            "RMSE": res["rmse"],
            "n_points": len(res["t_fit"]),
            "t_min": local_t_min,
            "t_max": local_t_max,
        })

    return pd.DataFrame(rows)


windows_to_test = [
    (0.30, 0.5),
    (0.20, 0.5),
    (0.35, 0.5),
    (0.25, 0.5),
    (0.4, 0.5),
]

window_rows = []

for start_frac, end_frac in windows_to_test:
    tmp = fit_one_fraction_window(
        front,
        method=main_method,
        start_fraction=start_frac,
        end_fraction=end_frac,
        exclude_boundary_hit=True,
    )
    window_rows.append(tmp)

window_df = pd.concat(window_rows, ignore_index=True)

window_summary = (
    window_df
    .groupby("window")
    .agg(
        mean_speed=("outward_speed", "mean"),
        left_speed=("outward_speed", lambda x: x.iloc[0]),
        right_speed=("outward_speed", lambda x: x.iloc[1]),
        min_R2=("R2", "min"),
        max_RMSE=("RMSE", "max"),
        n_points_min=("n_points", "min"),
    )
    .reset_index()
)

display(window_summary)

fig, ax = plt.subplots(figsize=(8, 4.8))

ax.plot(window_summary["window"], window_summary["mean_speed"], marker="o")
ax.set_xlabel("fit window")
ax.set_ylabel("mean outward speed")
ax.set_title(fr"Window sensitivity, method={main_method}")
plt.xticks(rotation=45)
fig.tight_layout()
plt.show()

In [ ]:
windows_to_test = [
    (0.60, 0.9),
    (0.70, 0.9),
    (0.75, 0.9),
    (0.80, 0.9),
    (0.85, 0.9),
]

window_rows = []

for start_frac, end_frac in windows_to_test:
    tmp = fit_one_fraction_window(
        front,
        method=main_method,
        start_fraction=start_frac,
        end_fraction=end_frac,
        exclude_boundary_hit=True,
    )
    window_rows.append(tmp)

window_df = pd.concat(window_rows, ignore_index=True)

window_summary = (
    window_df
    .groupby("window")
    .agg(
        mean_speed=("outward_speed", "mean"),
        left_speed=("outward_speed", lambda x: x.iloc[0]),
        right_speed=("outward_speed", lambda x: x.iloc[1]),
        min_R2=("R2", "min"),
        max_RMSE=("RMSE", "max"),
        n_points_min=("n_points", "min"),
    )
    .reset_index()
)

display(window_summary)

fig, ax = plt.subplots(figsize=(8, 4.8))

ax.plot(window_summary["window"], window_summary["mean_speed"], marker="o")
ax.set_xlabel("fit window")
ax.set_ylabel("mean outward speed")
ax.set_title(fr"Window sensitivity, method={main_method}")
plt.xticks(rotation=45)
fig.tight_layout()
plt.show()

## Thesis plots

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from front_analysis import get_run_files, read_front_file
from matplotlib.ticker import MaxNLocator


run_id_local = run_id          # or write 18 directly
method = "th_2"             # main threshold
fit_start_fraction = 0.70
fit_end_fraction   = 0.95
save_name = f"run_{run_id_local:03d}_{method}_fit_lr_thesis.png"

dat_file, front_file, rho_file = get_run_files(data_folder, param_base, run_id_local)
front_meta_local, front_local = read_front_file(front_file)

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 18,
    "axes.titlesize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

def set_max_ticks(ax, n=4, x=True, y=True):
    """Limit the number of major ticks on an axis."""
    if x:
        ax.xaxis.set_major_locator(MaxNLocator(nbins=n))
    if y:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=n))
        
t = np.asarray(front_local["time"], dtype=float)
t_min = fit_start_fraction * np.nanmax(t)
t_max = fit_end_fraction   * np.nanmax(t)

left_col  = f"x_left_{method}"
right_col = f"x_right_{method}"

def fit_and_plot(ax, x, side="left"):
    mask = np.isfinite(t) & np.isfinite(x) & (t >= t_min) & (t <= t_max)
    slope, intercept = np.polyfit(t[mask], x[mask], 1)

    v = -slope if side == "left" else slope
    v_label = "L" if side == "left" else "R"

    ax.plot(t, x, color="tab:blue", lw=1.8, label="front position")

    ax.plot(
        t[mask],
        slope * t[mask] + intercept,
        "--",
        color="tab:orange",
        lw=1.8,
        label=fr"linear fit ($v_{v_label}={v:.4f}$)"
    )

    ax.axvline(t_min, color="0.5", ls=":", lw=1.0)
    ax.axvline(t_max, color="0.5", ls=":", lw=1.0)

    set_max_ticks(ax, 4)
    ax.set_xlabel(r"time $t$")
    ax.set_ylabel(r"front position $x$")
    ax.set_title(f"{side} front")
    ax.legend(frameon=False, loc="best", fontsize=16)

    return slope, intercept, v

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.0), sharex=True)

slope_L, intercept_L, vL = fit_and_plot(axes[0], np.asarray(front_local[left_col], dtype=float), side="left")
slope_R, intercept_R, vR = fit_and_plot(axes[1], np.asarray(front_local[right_col], dtype=float), side="right")

axes[1].set_ylabel("")

fig.tight_layout()

save_folder = output_folder if "output_folder" in globals() else Path(".")
save_path = save_folder / save_name
fig.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved: {save_path}")
print(f"Left outward speed  vL = {vL:.6f}")
print(f"Right outward speed vR = {vR:.6f}")